## 6. Modelo Transformer preentrenado

Trabajo de María José Vire. Fase 1C: infraestructura y preanálisis del tokenizer.
El objetivo de esta fase es determinar la longitud máxima usando exclusivamente **train**.
Todavía no se ha realizado fine-tuning ni se presentan resultados de clasificación.

### 6.1 Definición del enfoque

Clasificación binaria de sentimiento en español con **BETO cased**, modelo base
`dccuchile/bert-base-spanish-wwm-cased`, y su tokenizer de Hugging Face.
La revisión del repositorio remoto queda fijada en la configuración.

- Única entrada: `comentario_limpio`.
- Etiqueta: `sentimiento_id`; Negativo = 0, Positivo = 1.
- `nombreAspecto`, `Valor`, `comentario` e `indice_original` no son predictores.
  `indice_original` se conserva únicamente para trazabilidad.
- Se reutilizan exactamente las particiones verificadas de Fase 0, comunes al baseline de Diego.
- Métrica principal futura: F1-macro. Train se utilizará para ajuste y validation para selección.
- Test permanece reservado para la fase final común; aquí solo se comprueba su integridad
  y su estructura. No se tokeniza, ni se examinan ejemplos para tomar decisiones.

**Entorno:** usar las dependencias existentes de Fase 0 y `requirements-transformer.txt`.
Transformers 4.57.1 publica compatibilidad con Python 3.12
([PyPI](https://pypi.org/project/transformers/4.57.1/)). La instalación se hace fuera de este
notebook; ninguna celda instala paquetes. PyTorch y la configuración CPU/GPU del entrenamiento
se decidirán posteriormente en Colab. Las semillas preparan reproducibilidad básica,
sin garantizar determinismo total.

In [1]:
from pathlib import Path
import sys

# Funciona desde la raíz o desde notebooks/, sin rutas personales.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/fase0.py").is_file()
             and (p / "configs/transformer.json").is_file()), None)
if root is None:
    raise RuntimeError("Abre el notebook dentro del repositorio con Fase 0 disponible.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
from IPython.display import display
from scripts.transformer import (
    load_config, set_seed, load_verified_partitions, load_tokenizer,
    tokenizer_metadata, analyze_train_lengths, record_max_length, save_analysis,
)

config_path = root / "configs/transformer.json"
config = load_config(config_path)
seed_status = set_seed(config["seed"])
print("Modelo:", config["model_id"])
print("Semillas:", seed_status)

Modelo: dccuchile/bert-base-spanish-wwm-cased
Semillas: {'seed': 42, 'python_random': True, 'numpy': True, 'torch': False}


### 6.2 Carga y verificación de las particiones

`load_verified_partitions()` utiliza exclusivamente `scripts.fase0.cargar_particiones()`.
Se comprueban hashes, tamaños, etiquetas, esquema y separación de las particiones mediante
la interfaz oficial. No se repiten la limpieza, deduplicación ni división del corpus.
Los siguientes recuentos son comprobaciones descriptivas de la interfaz, no métricas predictivas.

In [2]:
partitions = load_verified_partitions(root)
partition_summary = pd.DataFrame([
    {"partición": name, "observaciones": len(frame),
     "Negativo (0)": int(frame["sentimiento_id"].eq(0).sum()),
     "Positivo (1)": int(frame["sentimiento_id"].eq(1).sum())}
    for name, frame in partitions.items()
])
display(partition_summary)
for name, frame in partitions.items():
    print(name, "columnas:", list(frame.columns))
print("Tamaños esperados: train=16186, validation=3468, test=3469: verificados.")
print("Etiquetas {0, 1} e integridad Fase 0: verificadas.")
print("Entrada exclusiva:", config["text_column"], "| Etiqueta:", config["target_column"])
print("indice_original conservado únicamente para trazabilidad.")

,partición,observaciones,Negativo (0),Positivo (1)
0,train,16186,6870,9316
1,validation,3468,1472,1996
2,test,3469,1473,1996


train columnas: ['indice_original', 'comentario', 'Valor', 'nombreAspecto', 'comentario_limpio', 'sentimiento_id']
validation columnas: ['indice_original', 'comentario', 'Valor', 'nombreAspecto', 'comentario_limpio', 'sentimiento_id']
test columnas: ['indice_original', 'comentario', 'Valor', 'nombreAspecto', 'comentario_limpio', 'sentimiento_id']
Tamaños esperados: train=16186, validation=3468, test=3469: verificados.
Etiquetas {0, 1} e integridad Fase 0: verificadas.
Entrada exclusiva: comentario_limpio | Etiqueta: sentimiento_id
indice_original conservado únicamente para trazabilidad.


### 6.3 Selección y carga del tokenizer de BETO

Se usa `AutoTokenizer.from_pretrained` con el identificador y revisión de la configuración,
sin ejecutar código remoto. Se descargan exclusivamente archivos necesarios para tokenización;
**no se carga el clasificador ni pesos del modelo**. La clase y el vocabulario se registran
según el tokenizer realmente cargado, en lugar de inferirlos del nombre del modelo.

La caché se ubica en `results/transformer/tokenizer_cache/`, ignorada por Git.
La primera ejecución requiere acceso al Hub; las siguientes pueden usar la caché local.

In [3]:
tokenizer = load_tokenizer(config, root)
tokenizer_info = tokenizer_metadata(tokenizer, config)
display(pd.DataFrame([
    {"propiedad": key, "valor": str(value)}
    for key, value in tokenizer_info.items()
]))

/home/majovire/Escritorio/Visual Studio Code/Ciencia de Datos/Herramientas de IA/pln-sentimiento-estudiantil/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


,propiedad,valor
0,model_id,dccuchile/bert-base-spanish-wwm-cased
1,tokenizer_id,dccuchile/bert-base-spanish-wwm-cased
2,revision,c4d86612f51b4f46759c8390d1798c2febe71b93
3,class,BertTokenizerFast
4,is_fast,True
5,model_max_length,512
6,vocab_size,31002
7,vocab_with_added_tokens,31002
8,special_tokens,"{'unk_token': '[UNK]', 'sep_token': '[SEP]', '..."
9,special_token_ids,"{'[UNK]': 3, '[SEP]': 5, '[PAD]': 1, '[CLS]': ..."


### 6.4 Análisis de longitud tokenizada

La función vuelve a cargar las particiones por la interfaz oficial y extrae solamente **train**.
Tokeniza `comentario_limpio` sin truncamiento y sin padding, incluyendo los tokens especiales
que acompañarán cada secuencia individual. Los lotes de tokenización solo limitan memoria.
Los percentiles utilizan interpolación lineal de NumPy.

Regla fijada antes de observar las longitudes: elegir el menor candidato de
`[64, 128, 256, 512]` que cubra al menos el 99 % de train. Una secuencia de longitud igual
al límite está cubierta. Si ningún candidato cumple, elegir 512 e informar el porcentaje
que se truncaría. Validation y test no intervienen en esta decisión.

In [4]:
analysis = analyze_train_lengths(tokenizer, config, root)
print("Partición analizada:", analysis["split"], "| Observaciones:", analysis["n_observations"])
stat_names = {"min": "Mínimo", "mean": "Media", "median": "Mediana",
              "p90": "P90", "p95": "P95", "p99": "P99", "max": "Máximo"}
display(pd.DataFrame([
    {"estadístico": stat_names[key], "tokens": value}
    for key, value in analysis["statistics"].items()
]))
coverage_table = pd.DataFrame(analysis["coverage"]).rename(columns={
    "candidate": "límite candidato", "exceeds_count": "cantidad que excede",
    "exceeds_percent": "% que excede", "covered_percent": "% cubierto",
})
display(coverage_table)
print("max_length seleccionado:", analysis["max_length"])
print("Cobertura de al menos el 99 %:", analysis["coverage_target_met"])
print("Porcentaje que se truncaría:", analysis["selected_truncation_percent"])
if not analysis["coverage_target_met"]:
    print("Ningún candidato alcanza el 99 %; se utiliza el máximo permitido, 512.")

Token indices sequence length is longer than the specified maximum sequence length for this model (610 > 512). Running this sequence through the model will result in indexing errors


Partición analizada: train | Observaciones: 16186


,estadístico,tokens
0,Mínimo,3.000000
1,Media,20.284011
2,Mediana,14.000000
3,P90,38.000000
4,P95,52.000000
5,P99,103.150000
6,Máximo,610.000000


,límite candidato,cantidad que excede,% que excede,% cubierto
0,64,494,3.052020,96.947980
1,128,100,0.617818,99.382182
2,256,17,0.105029,99.894971
3,512,1,0.006178,99.993822


max_length seleccionado: 128
Cobertura de al menos el 99 %: True
Porcentaje que se truncaría: 0.6178178672927221


**Registro explícito de la decisión y del resumen descriptivo.** La siguiente celda actualiza
`configs/transformer.json` únicamente después del análisis real de train y guarda
`results/transformer/token_length_analysis.json`. El análisis por sí solo no modifica la
configuración. Se verifican el protocolo, la revisión del tokenizer y los hashes de train
antes de registrar la longitud. El resumen no incluye textos, predicciones ni métricas
 de clasificación.

Los resultados y la caché están ignorados por Git. Para reproducir desde la raíz:
`python -m scripts.transformer` (solo resumen); para registrar explícitamente la decisión:
`python -m scripts.transformer --record-max-length`. Si la caché está disponible, puede añadirse
`--local-files-only`. Este notebook también reproduce el análisis ejecutando sus celdas en orden.
Las particiones y metadatos locales de Fase 0 deben estar disponibles; no se regeneran aquí.

In [5]:
config = record_max_length(analysis, config_path, root)
analysis_path = save_analysis(analysis, root / "results/transformer/token_length_analysis.json")
print("Decisión registrada explícitamente:", config["max_length"])
print("Resumen descriptivo:", analysis_path.relative_to(root))

Decisión registrada explícitamente: 128
Resumen descriptivo: results/transformer/token_length_analysis.json


### 6.5 Preparación del protocolo experimental

Decisiones persistidas para el futuro fine-tuning:

- Seed: 42; máximo de 3 épocas.
- Learning rates a comparar: `1e-5`, `2e-5`, `3e-5`.
- Weight decay: 0.01; batch efectivo objetivo: 16.
- Evaluación al finalizar cada época, exclusivamente sobre validation.
- Selección: mayor F1-macro de validation; empate a cuatro decimales de F1-macro:
  mayor accuracy; si continúa, menor learning rate; si continúa, checkpoint de época más temprana.
- Test reservado para la fase final común.
- Longitud máxima derivada exclusivamente de train por la regla del 99 %.
- Padding dinámico por batch mediante `DataCollatorWithPadding` de Transformers.
  No se rellena todo el corpus hasta `max_length` en este preanálisis.

Siguen pendientes el batch físico y la acumulación de gradientes necesaria para alcanzar 16,
el dispositivo y precisión, las versiones de PyTorch/accelerate adecuadas al entorno,
y el registro/retención de checkpoints por época. Los checkpoints futuros se ubicarán en
`models/transformer/` y los resultados en `results/transformer/`; ambas rutas están ignoradas
por Git. El baseline de Diego permanece en su notebook y artefactos originales. La comparación
final se preparará en una fase posterior.

**Estado actual: no se ha cargado ni entrenado ningún clasificador Transformer; no hay
predicciones, pérdidas ni métricas de clasificación de este modelo.**